# 02 — EDA on the chair splats

This is not a generic "plot every column" pass. Every section answers a design
question you have to settle before writing model code:

| § | Question | Decides |
|---|---|---|
| 3 | How many Gaussians per object, and how variable? | fixed-size vs variable-size representation |
| 4 | Are objects already centred and scaled consistently? | the normalisation step |
| 5 | What does the scale distribution look like? | parameterisation (log vs linear) |
| 6 | How much opacity mass is prunable? | how cheaply you can hit a fixed N |
| 7 | Do the 45 SH `f_rest` coefficients carry information? | **59 vs 14 channels per Gaussian** |
| 8 | Are quaternions canonicalised? | whether the rotation target is well-posed |
| 9 | How densely do Gaussians pack into a voxel grid? | grid resolution for a GaussianCube-style model |
| 10 | Which attributes correlate? | whether joint modelling buys anything |

Section 11 writes out normalisation constants; section 12 collects the findings.

## 1. Setup

In [ ]:
import os, sys, time, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
import splatlib as sl

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 9})

CANDIDATES = [
    "/lustre/data/mls04/ModelNet_Splats/extracted/train",
    "/data/ModelNet_Splats/extracted/train",
    os.environ.get("DATASET_DIR", "") + "/extracted/train",
]
DATA_ROOT = next((Path(c) for c in CANDIDATES if c and Path(c).is_dir()), None)
assert DATA_ROOT is not None, "set DATA_ROOT manually"

CATEGORY  = "chair"
OUT       = Path.home() / "psiml" / "outputs";  OUT.mkdir(parents=True, exist_ok=True)
CACHE     = OUT / f"{CATEGORY}_stats.parquet"
SAMPLE_NPZ = OUT / f"{CATEGORY}_sample.npz"

files = sl.find_objects(DATA_ROOT, category=CATEGORY)
print(f"{len(files)} {CATEGORY} objects at {DATA_ROOT}")

## 2. Build the stats table

One pass over all objects, cached to parquet. On Lustre the bottleneck is
per-file metadata latency, not bandwidth — expect a few minutes the first time
and instant afterwards.

If it is painfully slow, that is your first concrete signal that you need the
sharded preprocessing described at the end.

In [ ]:
if CACHE.exists():
    df = pd.read_parquet(CACHE)
    print(f"loaded cached stats for {len(df)} objects")
else:
    rows, t0, bad = [], time.time(), []
    for i, f in enumerate(files):
        try:
            rows.append(sl.object_stats(sl.load_ply(f)))
        except Exception as e:
            bad.append((f.name, repr(e)))
        if (i + 1) % 100 == 0:
            el = time.time() - t0
            print(f"  {i+1}/{len(files)}  {el:.0f}s elapsed  "
                  f"~{el/(i+1)*(len(files)-i-1):.0f}s left")
    df = pd.DataFrame(rows)
    df.to_parquet(CACHE)
    print(f"\n{len(df)} objects in {time.time()-t0:.0f}s -> {CACHE}")
    if bad:
        print(f"{len(bad)} FAILED:"); [print("   ", *b) for b in bad[:10]]

df.head()

In [ ]:
# Pooled per-Gaussian sample. Object-level means hide within-object structure;
# this gives the population distribution without loading 889 files into RAM.
if SAMPLE_NPZ.exists():
    S = dict(np.load(SAMPLE_NPZ))
    print("loaded cached sample")
else:
    rng = np.random.default_rng(0)
    subset = [files[i] for i in sorted(rng.choice(len(files),
                                                  min(200, len(files)), replace=False))]
    t0 = time.time()
    S = sl.gaussian_sample(subset, per_file=3000, seed=0)
    np.savez_compressed(SAMPLE_NPZ, **S)
    print(f"sampled {len(S['xyz']):,} gaussians from {len(subset)} objects "
          f"in {time.time()-t0:.0f}s")

print({k: v.shape for k, v in S.items()})

## 3. How many Gaussians per object?

The single most consequential number. Diffusion models want a **fixed-size**
tensor. If N varies a lot you must either pad and mask (transformer / point
approach) or resample to a fixed count (GaussianCube's grid approach).

The reference line at 32 768 = 32³ is GaussianCube's structured-grid capacity.
Whether your objects sit above or below it decides whether their voxelisation
step is mostly *dropping* Gaussians or mostly *padding* empty voxels.

In [ ]:
n = df["n"].values
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))

ax[0].hist(n, bins=50, color="#4C72B0", edgecolor="white", lw=0.4)
for v, c, s in [(32768, "crimson", "32³ = 32768"), (16384, "orange", "16³·4")]:
    if n.min() < v < n.max() * 1.4:
        ax[0].axvline(v, color=c, ls="--", lw=1.2, label=s)
ax[0].set(xlabel="gaussians per object", ylabel="objects", title="count distribution")
ax[0].legend(fontsize=7)

ax[1].plot(np.sort(n), np.linspace(0, 1, len(n)), lw=1.6)
ax[1].axhline(0.5, color="grey", lw=0.6)
ax[1].set(xlabel="gaussians", ylabel="cumulative fraction", title="CDF")

ax[2].scatter(df["radius_max"], n, s=5, alpha=0.4)
ax[2].set(xlabel="object radius", ylabel="gaussians",
          title="count vs object size")
plt.tight_layout(); plt.show()

q = np.percentile(n, [0, 1, 25, 50, 75, 99, 100]).astype(int)
print(f"min {q[0]:,}   p1 {q[1]:,}   p25 {q[2]:,}   median {q[3]:,}   "
      f"p75 {q[4]:,}   p99 {q[5]:,}   max {q[6]:,}")
print(f"total across category: {n.sum():,}")
print(f"spread (max/min): {n.max()/max(n.min(),1):.1f}x")
print(f"objects above 32768: {(n > 32768).sum()} / {len(n)} ({(n>32768).mean():.1%})")

## 4. Are objects already normalised?

Diffusion needs inputs on a consistent, roughly unit scale. Two things to check:

- **Centring** — is each object's centre at the origin?
- **Extent** — is each object's radius the same, or does it vary per object?

If radius varies, you must normalise per object, and then decide whether the
model should *predict* the original scale or ignore it. For unconditional shape
generation, ignoring it is standard.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))

ax[0].scatter(df["cx"], df["cy"], s=5, alpha=0.4, label="x vs y")
ax[0].scatter(df["cx"], df["cz"], s=5, alpha=0.4, label="x vs z")
ax[0].axhline(0, color="k", lw=0.5); ax[0].axvline(0, color="k", lw=0.5)
ax[0].set(xlabel="centre x", ylabel="centre y / z", title="bbox centre")
ax[0].legend(fontsize=7)

ax[1].hist(df["radius_max"], bins=45, color="#55A868", edgecolor="white", lw=0.4)
ax[1].set(xlabel="max radius from bbox centre", ylabel="objects", title="object size")

for c, lab in [("ext_x", "x"), ("ext_y", "y"), ("ext_z", "z")]:
    ax[2].hist(df[c], bins=45, alpha=0.55, label=lab)
ax[2].set(xlabel="bbox extent", title="extent per axis"); ax[2].legend(fontsize=7)

ax[3].hist(df["centroid_off"] / df["radius_max"], bins=45,
           color="#C44E52", edgecolor="white", lw=0.4)
ax[3].set(xlabel="|centroid - bbox centre| / radius",
          title="mass asymmetry")
plt.tight_layout(); plt.show()

print("bbox centre     :", df[["cx","cy","cz"]].abs().max().round(4).to_dict(),
      " (max |value|)")
print(f"radius          : mean {df['radius_max'].mean():.4f} "
      f"std {df['radius_max'].std():.4f}  "
      f"CV {df['radius_max'].std()/df['radius_max'].mean():.3f}")
print(f"radius range    : [{df['radius_max'].min():.4f}, {df['radius_max'].max():.4f}]")
centred = df[["cx","cy","cz"]].abs().max().max() < 0.02 * df["radius_max"].mean()
uniform = df["radius_max"].std() / df["radius_max"].mean() < 0.02
print(f"\n=> pre-centred:   {centred}")
print(f"=> uniform scale: {uniform}")
print("   (if either is False, normalise per object before training)")

The largest-extent axis across the whole category also confirms the up-axis
finding from notebook 01 — chairs are taller than they are deep, so the vertical
axis should dominate consistently.

In [ ]:
dom = df[["ext_x","ext_y","ext_z"]].values.argmax(1)
lab, cnt = np.unique(dom, return_counts=True)
for l, c in zip(lab, cnt):
    print(f"  largest extent along {'xyz'[l]}: {c:4d} objects ({c/len(df):.1%})")
print("\nA clean majority on one axis => orientation is consistent across the dataset.")

## 5. Scale

Scales are stored as `log(s)` and that is almost certainly the right space to
model in: it makes the distribution roughly Gaussian, keeps scales positive by
construction, and avoids the model having to learn a hard positivity constraint.

**Anisotropy** = longest axis / shortest axis. 3DGS optimisation tends to
produce flat, disc-like Gaussians that hug surfaces. Extreme outliers (ratios in
the hundreds) are numerically nasty and worth clamping.

In [ ]:
ls = S["raw_scale"].ravel()
sc = S["scale"]
ssort = np.sort(sc, axis=1)
aniso = ssort[:, 2] / np.maximum(ssort[:, 0], 1e-12)

fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))

ax[0].hist(ls, bins=120, color="#4C72B0", density=True)
mu, sd = ls.mean(), ls.std()
xs = np.linspace(ls.min(), ls.max(), 200)
ax[0].plot(xs, np.exp(-0.5*((xs-mu)/sd)**2)/(sd*np.sqrt(2*np.pi)),
           "r--", lw=1.3, label=f"N({mu:.2f}, {sd:.2f}²)")
ax[0].set(xlabel="log(scale)", title="log-scale (all axes)"); ax[0].legend(fontsize=7)

for i, c in enumerate("xyz"):
    ax[1].hist(S["raw_scale"][:, i], bins=90, alpha=0.5, density=True, label=f"axis {c}")
ax[1].set(xlabel="log(scale)", title="per axis"); ax[1].legend(fontsize=7)

ax[2].hist(np.log10(aniso), bins=100, color="#C44E52", density=True)
for v in [10, 100]:
    ax[2].axvline(np.log10(v), color="k", ls="--", lw=0.8)
ax[2].set(xlabel="log10(s_max / s_min)", title="anisotropy")

ax[3].hist(np.log10(ssort[:, 1] / np.maximum(ssort[:, 0], 1e-12)), bins=100,
           alpha=0.6, label="mid/min")
ax[3].hist(np.log10(ssort[:, 2] / np.maximum(ssort[:, 1], 1e-12)), bins=100,
           alpha=0.6, label="max/mid")
ax[3].set(xlabel="log10 ratio", title="shape: needle vs disc"); ax[3].legend(fontsize=7)
plt.tight_layout(); plt.show()

from scipy import stats as st
print(f"log-scale       mean {mu:.3f}  std {sd:.3f}  "
      f"skew {st.skew(ls):.3f}  kurtosis {st.kurtosis(ls):.3f}")
print(f"scale (linear)  median {np.median(sc):.5f}  "
      f"p99 {np.percentile(sc, 99):.5f}  max {sc.max():.5f}")
print(f"anisotropy      median {np.median(aniso):.2f}  "
      f"p95 {np.percentile(aniso, 95):.1f}  p99.9 {np.percentile(aniso, 99.9):.1f}")
print(f"                >10x: {(aniso>10).mean():.2%}   >100x: {(aniso>100).mean():.3%}")
print(f"\nmedian scale / median object radius = "
      f"{np.median(sc)/df['radius_max'].median():.5f}")

## 6. Opacity, and how much you can prune

3DGS optimisation leaves a long tail of near-transparent Gaussians that
contribute almost nothing to any render. Pruning them is close to free and is
the cheapest way to reach a fixed N.

The honest way to measure "contributes nothing" is not opacity alone — a large
faint Gaussian can matter more than a small opaque one. A reasonable importance
proxy is $\alpha \cdot s_x s_y s_z$ (opacity × ellipsoid volume). The second
panel shows what fraction of *total importance* the discarded Gaussians hold.

In [ ]:
op = S["opacity"]
vol = np.prod(S["scale"], axis=1)
imp = op * vol
imp_sorted = np.sort(imp)[::-1]
imp_cum = np.cumsum(imp_sorted) / imp_sorted.sum()

fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))

ax[0].hist(op, bins=100, color="#4C72B0")
ax[0].set(xlabel="opacity (sigmoid applied)", ylabel="gaussians", title="opacity")

ax[1].hist(S["raw_opacity"], bins=100, color="#8172B2")
ax[1].set(xlabel="logit(opacity) as stored", title="raw opacity")

thr = np.array([0.005, 0.01, 0.02, 0.05, 0.1, 0.2])
frac_drop = [(op < t).mean() for t in thr]
imp_lost = [imp[op < t].sum() / imp.sum() for t in thr]
ax[2].plot(thr, frac_drop, "o-", label="fraction of gaussians dropped")
ax[2].plot(thr, imp_lost, "s-", label="fraction of importance lost")
ax[2].set(xlabel="opacity threshold", xscale="log", title="pruning trade-off")
ax[2].legend(fontsize=7)

k = np.arange(1, len(imp_cum) + 1) / len(imp_cum)
ax[3].plot(k, imp_cum, lw=1.6)
for f in [0.5, 0.25]:
    j = int(f * len(imp_cum)) - 1
    ax[3].plot(f, imp_cum[j], "ro", ms=4)
    ax[3].annotate(f"top {f:.0%} -> {imp_cum[j]:.1%}", (f, imp_cum[j]),
                   textcoords="offset points", xytext=(6, -10), fontsize=7)
ax[3].set(xlabel="fraction kept (sorted by importance)",
          ylabel="cumulative importance", title="importance concentration")
plt.tight_layout(); plt.show()

print(f"opacity   mean {op.mean():.3f}  median {np.median(op):.3f}")
print(f"{'thresh':>8}{'dropped':>10}{'importance lost':>18}")
for t, d, l in zip(thr, frac_drop, imp_lost):
    print(f"{t:>8.3f}{d:>10.2%}{l:>18.3%}")
for f in [0.75, 0.5, 0.25]:
    print(f"keeping top {f:.0%} by importance retains "
          f"{imp_cum[int(f*len(imp_cum))-1]:.2%} of it")

In [ ]:
# Per-object view: does the prunable fraction vary a lot between objects?
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
ax[0].hist(df["frac_op_lt01"], bins=45, color="#4C72B0")
ax[0].set(xlabel="fraction with opacity < 0.01", ylabel="objects", title="per object")
ax[1].scatter(df["n"], df["frac_op_lt05"], s=5, alpha=0.4)
ax[1].set(xlabel="gaussians", ylabel="fraction opacity < 0.05",
          title="more gaussians => more junk?")
eff = df["n"] * (1 - df["frac_op_lt01"])
ax[2].hist(eff, bins=45, color="#55A868")
ax[2].axvline(32768, color="crimson", ls="--", lw=1.2, label="32768")
ax[2].set(xlabel="effective gaussians (opacity ≥ 0.01)", title="after pruning")
ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

print(f"effective count: median {eff.median():,.0f}  "
      f"min {eff.min():,.0f}  max {eff.max():,.0f}")
print(f"above 32768 after pruning: {(eff > 32768).mean():.1%} of objects")

## 7. Colour and spherical harmonics — the dimensionality question

This section can halve your model. ModelNet CAD meshes are **untextured**, so
the splats were fitted to shaded renders of grey geometry. Two consequences to
test:

1. **Is the colour nearly constant?** If so, colour carries almost no shape
   information and can be dropped or heavily compressed.
2. **Do the 45 `f_rest` coefficients carry energy?** They encode view-dependent
   appearance — specularity, really. On matte untextured geometry they may be
   near zero.

Per-Gaussian dimensionality: 3 (xyz) + 3 (scale) + 4 (quat) + 1 (opacity)
+ 3 (DC) = **14**, versus **59** if you keep all SH. For a diffusion model that
is a 4× difference in output channels.

In [ ]:
rgb = S["rgb"]
sat = rgb.max(1) - rgb.min(1)          # 0 = perfectly grey
lum = rgb.mean(1)

fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))
for i, c in enumerate(["red", "green", "blue"]):
    ax[0].hist(rgb[:, i], bins=80, alpha=0.5, color=c, label=c[0].upper())
ax[0].set(xlabel="channel value", title="RGB from SH DC"); ax[0].legend(fontsize=7)

ax[1].hist(sat, bins=90, color="#8172B2")
ax[1].set(xlabel="max-min across channels", title="saturation (0 = grey)")

ax[2].hist(lum, bins=90, color="#937860")
ax[2].set(xlabel="mean(RGB)", title="luminance")

ax[3].hist(df["rest_energy_ratio"], bins=45, color="#C44E52")
ax[3].set(xlabel="sum(f_rest²) / sum(f_dc²)", title="SH rest energy, per object")
plt.tight_layout(); plt.show()

print(f"saturation     median {np.median(sat):.4f}   p95 {np.percentile(sat, 95):.4f}")
print(f"fraction with saturation < 0.05 (near-grey): {(sat < 0.05).mean():.1%}")
print(f"luminance      mean {lum.mean():.3f}  std {lum.std():.3f}")
print(f"\nSH rest / DC energy ratio: median {df['rest_energy_ratio'].median():.4f}  "
      f"max {df['rest_energy_ratio'].max():.4f}")
print(f"mean |f_rest|            : {df['rest_absmean'].mean():.5f}")

if df["rest_energy_ratio"].median() < 0.02:
    print("\n=> f_rest is essentially empty. DROP IT: 59 -> 14 channels per gaussian.")
elif df["rest_energy_ratio"].median() < 0.15:
    print("\n=> f_rest is weak. Try dropping it and compare renders; likely safe.")
else:
    print("\n=> f_rest carries real energy; keep at least SH degree 1.")

if np.median(sat) < 0.05:
    print("=> colour is near-greyscale. Consider 1 luminance channel instead of 3.")

Before acting on that, confirm it visually: re-render one object with `f_rest`
zeroed. If the images are indistinguishable, the coefficients are noise and
dropping them costs nothing.

In [ ]:
sp = sl.load_ply(files[0])
sp_nosh = sl.load_ply(files[0]); sp_nosh.f_rest = np.zeros((len(sp_nosh), 0), np.float32)

fig, ax = plt.subplots(1, 3, figsize=(12, 4.2))
a = sl.render(sp,      azim=40, elev=15, H=300, W=300, up_axis=1)
b = sl.render(sp_nosh, azim=40, elev=15, H=300, W=300, up_axis=1)
for axi, im, t in zip(ax, [a, b, np.abs(a - b).mean(-1)],
                      ["full SH", "DC only", "|difference|"]):
    axi.imshow(im if im.ndim == 3 else im, cmap=None if im.ndim == 3 else "inferno")
    axi.set_title(t); axi.axis("off")
plt.tight_layout(); plt.show()
print(f"mean abs difference: {np.abs(a-b).mean():.5f}   max: {np.abs(a-b).max():.5f}")
print("(our renderer uses only the DC term anyway, so this is a lower bound --\n"
      " for a real check, render with gsplat at a few viewing angles.)")

## 8. Rotation — the double-cover problem

A quaternion $q$ and its negation $-q$ describe the **same rotation**. If the
dataset is not canonicalised, identical geometry appears under two different
targets, and any model trained to regress or denoise raw quaternions gets
inconsistent supervision.

The fix is one line — force $w \ge 0$ — but you have to know whether it is
needed. If the fraction with $w < 0$ is near 50%, the data is uncanonicalised.

In [ ]:
q = S["quat"]
qc = sl.canonicalize_quat(q)

fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))
for i, lab in enumerate("wxyz"):
    ax[0].hist(q[:, i], bins=80, alpha=0.45, label=lab)
ax[0].set(xlabel="component", title="raw quaternion"); ax[0].legend(fontsize=7)

for i, lab in enumerate("wxyz"):
    ax[1].hist(qc[:, i], bins=80, alpha=0.45, label=lab)
ax[1].set(xlabel="component", title="after w >= 0"); ax[1].legend(fontsize=7)

ax[2].hist(df["frac_qw_neg"], bins=45, color="#C44E52")
ax[2].axvline(0.5, color="k", ls="--", lw=1)
ax[2].set(xlabel="fraction with w < 0", ylabel="objects", title="per object")

ang = 2 * np.degrees(np.arccos(np.clip(np.abs(q[:, 0]), 0, 1)))
ax[3].hist(ang, bins=80, color="#55A868")
ax[3].set(xlabel="rotation angle (deg)", title="magnitude of rotation")
plt.tight_layout(); plt.show()

print(f"fraction with w < 0: {(q[:,0] < 0).mean():.3f}  (0.5 => not canonicalised)")
print(f"mean |q| = {np.linalg.norm(q, axis=1).mean():.6f}  (should be 1)")
print(f"rotation angle: median {np.median(ang):.1f}°  "
      f"uniform-random would be ~126°")
if abs((q[:,0] < 0).mean() - 0.5) < 0.1:
    print("\n=> NOT canonicalised. Apply canonicalize_quat() in your dataloader.")

## 9. Spatial structure — can this go on a grid?

GaussianCube's core move is mapping an unordered Gaussian set onto a fixed
voxel grid so that a standard 3D U-Net can be used. Whether that is comfortable
or lossy depends on two numbers:

- **occupancy** — what fraction of voxels contain anything. Very low occupancy
  means a 3D grid wastes most of its capacity on empty space.
- **collisions** — how many Gaussians land in the same voxel. High collision
  counts mean the mapping must merge or discard.

Nearest-neighbour distance between centres tells you the natural length scale,
and therefore the resolution below which you are definitely losing detail.

In [ ]:
from scipy.spatial import cKDTree

sp = sl.load_ply(files[0])
p = sp.xyz - 0.5 * (sp.xyz.min(0) + sp.xyz.max(0))
p = p / np.abs(p).max()                      # -> [-1,1]

tree = cKDTree(p)
d, _ = tree.query(p, k=2)
nnd = d[:, 1]

fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.3))
ax[0].hist(nnd, bins=100, color="#4C72B0")
for res in [32, 64, 128]:
    ax[0].axvline(2 / res, ls="--", lw=1, label=f"{res}³ voxel")
ax[0].set(xlabel="nearest-neighbour distance (normalised)", title="packing density")
ax[0].legend(fontsize=7)

rows = [sl.voxel_occupancy(sp.xyz, r) for r in [16, 32, 64, 128]]
vo = pd.DataFrame(rows)
ax[1].plot(vo["res"], vo["occupancy_frac"], "o-", label="occupied fraction")
ax[1].plot(vo["res"], vo["frac_in_singleton"], "s-", label="gaussians alone in voxel")
ax[1].set(xlabel="grid resolution", title="grid utilisation")
ax[1].set_xscale("log", base=2)
ax[1].legend(fontsize=7)

ax[2].plot(vo["res"], vo["per_voxel_max"], "o-", label="max per voxel")
ax[2].plot(vo["res"], vo["per_voxel_mean"], "s-", label="mean per occupied voxel")
ax[2].set(xlabel="grid resolution", yscale="log", title="collisions")
ax[2].set_xscale("log", base=2)
ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

print(f"nearest-neighbour distance: median {np.median(nnd):.5f}  "
      f"p95 {np.percentile(nnd, 95):.5f}")
print(f"a 64³ voxel is {2/64:.5f} wide\n")
print(vo[["res","n_gaussians","occupied","occupancy_frac",
          "per_voxel_mean","per_voxel_max","frac_in_singleton"]].round(4).to_string(index=False))

In [ ]:
# Same thing averaged over several objects, so you are not tuning to one chair.
rng = np.random.default_rng(3)
sub = [files[i] for i in rng.choice(len(files), 25, replace=False)]
agg = pd.DataFrame([{**sl.voxel_occupancy(sl.load_ply(f).xyz, r), "res": r}
                    for f in sub for r in [32, 64, 128]])
print(agg.groupby("res")[["occupancy_frac", "capacity_ratio",
                          "per_voxel_mean", "per_voxel_max",
                          "frac_in_singleton"]].mean().round(4).to_string())
print("\ncapacity_ratio = N gaussians / res³. Above 1 means the grid cannot hold")
print("them all even in principle; well below 1 means most voxels sit empty.")

## 10. Correlations

If attributes are strongly coupled — say small Gaussians are systematically more
opaque — a model that treats channels independently is leaving structure on the
table, and a factorised prior will be wrong.

In [ ]:
feat = pd.DataFrame({
    "x": S["xyz"][:, 0], "y": S["xyz"][:, 1], "z": S["xyz"][:, 2],
    "log_s_mean": S["raw_scale"].mean(1),
    "log_s_spread": S["raw_scale"].std(1),
    "logit_opacity": S["raw_opacity"],
    "luminance": S["rgb"].mean(1),
    "saturation": S["rgb"].max(1) - S["rgb"].min(1),
    "radius": np.linalg.norm(S["xyz"] - S["xyz"].mean(0), axis=1),
})
corr = feat.corr()

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6),
                       gridspec_kw={"width_ratios": [1.15, 1]})
im = ax[0].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax[0].set_xticks(range(len(corr))); ax[0].set_xticklabels(corr.columns, rotation=55,
                                                         ha="right", fontsize=7.5)
ax[0].set_yticks(range(len(corr))); ax[0].set_yticklabels(corr.columns, fontsize=7.5)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax[0].text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center",
                   fontsize=6.5, color="white" if abs(corr.iloc[i,j]) > 0.55 else "black")
ax[0].set_title("Pearson correlation"); ax[0].grid(False)
plt.colorbar(im, ax=ax[0], fraction=0.045)

hb = ax[1].hexbin(S["raw_scale"].mean(1), S["raw_opacity"],
                  gridsize=60, bins="log", cmap="viridis")
ax[1].set(xlabel="mean log(scale)", ylabel="logit(opacity)",
          title="scale vs opacity")
plt.colorbar(hb, ax=ax[1], fraction=0.045, label="log10 count")
plt.tight_layout(); plt.show()

strong = [(a, b, corr.loc[a, b]) for i, a in enumerate(corr.columns)
          for b in corr.columns[i+1:] if abs(corr.loc[a, b]) > 0.25]
print("|r| > 0.25:")
for a, b, r in sorted(strong, key=lambda t: -abs(t[2])):
    print(f"  {a:<15} {b:<15} {r:+.3f}")

## 11. Normalisation constants

Diffusion models assume inputs roughly at zero mean, unit variance — the noise
schedule is calibrated for it. Compute the constants once, save them, and use
the same file for training and sampling. Recomputing them per run is a classic
source of silent train/inference mismatch.

In [ ]:
stats = sl.save_norm_stats(S, OUT / f"{CATEGORY}_norm_stats.json")
print(json.dumps(stats, indent=2))
print(f"\nsaved to {OUT / f'{CATEGORY}_norm_stats.json'}")

In [ ]:
# Verify: after normalising, everything should sit near mean 0 / std 1.
def normalise(S, st):
    return np.concatenate([
        (S["xyz"] - st["xyz"]["mean"]) / st["xyz"]["std"],
        (S["raw_scale"] - st["log_scale"]["mean"]) / st["log_scale"]["std"],
        ((S["raw_opacity"] - st["logit_opacity"]["mean"])
         / st["logit_opacity"]["std"])[:, None],
        sl.canonicalize_quat(S["quat"]),
        (S["rgb"] - st["rgb"]["mean"]) / st["rgb"]["std"],
    ], 1)

X = normalise(S, stats)
names = ["x","y","z","ls0","ls1","ls2","opa","qw","qx","qy","qz","r","g","b"]
print(f"normalised tensor: {X.shape}  ({X.shape[1]} channels per gaussian)\n")
print(f"{'ch':<6}{'mean':>9}{'std':>9}{'min':>9}{'max':>9}")
for i, nm in enumerate(names):
    c = X[:, i]
    print(f"{nm:<6}{c.mean():>9.3f}{c.std():>9.3f}{c.min():>9.2f}{c.max():>9.2f}")
print("\nChannels with |min| or max beyond ~6 have heavy tails -- consider")
print("clipping or a per-channel quantile transform before training.")

## 12. Findings

Fill this in from your own numbers — it is what you will show the mentors.

In [ ]:
d = dict(
    n_obj=len(df),
    n_med=df["n"].median(), n_min=df["n"].min(), n_max=df["n"].max(),
    over32k=(df["n"] > 32768).mean(),
    ctr=df[["cx","cy","cz"]].abs().max().max(),
    rad=df["radius_max"].mean(),
    cv=df["radius_max"].std() / df["radius_max"].mean(),
    axis="xyz"[int(np.bincount(df[["ext_x","ext_y","ext_z"]].values.argmax(1)).argmax())],
    ls_m=S["raw_scale"].mean(), ls_s=S["raw_scale"].std(),
    ani=df["aniso_med"].median(), ani99=df["aniso_p99"].median(),
    op=S["opacity"].mean(),
    op01=(S["opacity"] < 0.01).mean(), op05=(S["opacity"] < 0.05).mean(),
    sat=np.median(S["rgb"].max(1) - S["rgb"].min(1)),
    rest=df["rest_energy_ratio"].median(),
    qneg=(S["quat"][:, 0] < 0).mean(),
)

print(f"DATASET      {CATEGORY}, {d['n_obj']} objects")
print()
print(f"SIZE         median {d['n_med']:,.0f} gaussians   "
      f"range [{d['n_min']:,} .. {d['n_max']:,}]")
print(f"             {d['over32k']:.0%} of objects exceed 32768 (32^3)")
print()
print(f"GEOMETRY     bbox centre, max |offset|  {d['ctr']:.4f}")
print(f"             radius mean {d['rad']:.4f}   CV {d['cv']:.3f}")
print(f"             dominant axis: {d['axis']}")
print()
print(f"SCALE        log-scale mean {d['ls_m']:.3f}  std {d['ls_s']:.3f}")
print(f"             anisotropy median {d['ani']:.2f}   p99 {d['ani99']:.1f}")
print()
print(f"OPACITY      mean {d['op']:.3f}")
print(f"             below 0.01: {d['op01']:.1%}    below 0.05: {d['op05']:.1%}")
print()
print(f"APPEARANCE   saturation median {d['sat']:.4f}")
print(f"             SH rest/DC energy median {d['rest']:.4f}")
print(f"             => per-gaussian channels: "
      f"{'14 (drop f_rest)' if d['rest'] < 0.05 else '59 (keep SH)'}")
print()
print(f"ROTATION     fraction w<0: {d['qneg']:.3f}  "
      f"{'-> canonicalise' if abs(d['qneg'] - 0.5) < 0.1 else '-> already canonical'}")

---

## Suggested next steps

**Immediately, before any model code:**

1. **Preprocess into shards.** Lustre handles a few large sequential reads far
   better than 889 small random ones. Convert the category into a handful of
   `.npy` files holding pre-parsed, pre-normalised, fixed-N tensors. This is the
   single biggest wall-clock win available to you, and you need the fixed-N step
   anyway. See `preprocess.py`.

2. **Settle the representation.** Two families, and the numbers above should
   decide it:
   - *Set-based* — treat the object as an unordered set, use a transformer with
     attention, pad/mask to a fixed N. Handles variable counts naturally, no
     voxelisation loss, but $O(N^2)$ attention over tens of thousands of tokens
     is expensive.
   - *Grid-based* (GaussianCube) — map to a fixed voxel grid via optimal
     transport, then a standard 3D U-Net. Reuses all of 2D/3D diffusion
     tooling. Your §9 occupancy numbers tell you the resolution.

   Given five days and one A100, the grid approach reuses more existing code.

3. **Fix N.** Sort by importance ($\alpha \cdot s_xs_ys_z$), take the top N,
   pad with zero-opacity Gaussians if short. §6 tells you what that costs.

4. **Build the renderer-based eval now, not later.** A loss going down tells you
   nothing about whether samples look like chairs. Being able to render a batch
   to a PNG grid every K steps is worth more than any metric. For real
   view-dependent rendering, `pip install gsplat` — you have `nvcc` 12.8, so a
   source build will work if no wheel matches.

**For the model itself:**

5. **Start small deliberately.** Train on 64 chairs at low N first. If it cannot
   overfit 64 objects, it will not fit 889, and you find that out in minutes
   instead of hours.

6. **Predict in normalised, activated-inverse space** — log-scale, logit-opacity,
   canonicalised quaternion. Apply activations only when rendering.

7. **Watch the quaternions.** Even canonicalised, the unit-norm constraint is
   awkward for diffusion. Two common workarounds: predict a 6D rotation
   representation (continuous, no double cover) and orthogonalise, or predict
   the covariance's Cholesky factor directly and skip rotations entirely.

**Worth checking if time allows:**

8. Is chair-only enough, or do you want a second category for a conditional
   model? A class-conditional model on 3–5 categories is a natural "extension".

9. How well does a trivial baseline do — nearest neighbour retrieval from the
   training set? If your generated samples cannot beat retrieval on a perceptual
   metric, that is worth knowing before the final presentation rather than
   during it.